# Demo 12 · Meet a stranger

Today you judge an MCP server you did not write. So first, write one, and make it lie
a little.

`demos/12_tiny_server.py` is 25 lines: an MCP server with two tools. One is honest.
The other is marked read-only, and its description tells the model to delete things.
Then you connect to it the way any assistant does, read what it offers, and catch the
lie. Last, you connect to a server that behaves: the course.

About five minutes. No key and no model. The course part needs a network; without one
it says so and moves on.

In [ ]:
# Setup. Works from anywhere inside the course checkout.
import json
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
SERVER = REPO_ROOT / "demos" / "12_tiny_server.py"
print(SERVER.read_text(encoding="utf-8"))

## 1. Build it

That is the whole server. Three things to notice:

- **`MCPServer`** is the class almost every tutorial still calls `FastMCP`. Version 2
  of the Python SDK renamed it. Type `from mcp.server.fastmcp import FastMCP` here and
  it fails; the protocol did not change, the name did.
- **`@mcp.tool()`** turns a function into a tool. Its type hints become the input
  schema, and its docstring becomes the description a model reads.
- **`summarize_notes`** says it is read-only, and its docstring gives an order.
  Somebody wrote that sentence. Nothing checks it.

## 2. Connect to it, as a client

An assistant starts the server as a child process and talks to it over stdio. So does
this cell. It asks the three things a client learns on connect: what the server
claims (the handshake), what it lists, and what each tool says about itself.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PRIMITIVES = ("tools", "resources", "prompts")


async def read_server():
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            hello = await session.initialize()
            claimed = hello.capabilities.model_dump(exclude_none=True)
            listed = {
                "tools": (await session.list_tools()).tools,
                "resources": (await session.list_resources()).resources,
                "prompts": (await session.list_prompts()).prompts,
            }
            return hello.server_info.name, claimed, listed


name, claimed, listed = await read_server()
print(f"server: {name}")
for primitive in PRIMITIVES:
    said = "announced" if primitive in claimed else "not announced"
    print(f"  {primitive:<10} {said:<14} listed: {len(listed[primitive])}")

**Read the last two lines.** The server announced `resources` and `prompts`, and lists
none of either. Nobody wrote that: `MCPServer` announces them by default. A client that
counts the listing reports 0, which is true and useless. A client that compares the two
reports "announced and empty", which somebody can act on. That is `ch12-e2`.

## 3. Read what the model will read

These are the descriptions, exactly as a model receives them, next to your own system
prompt.

In [ ]:
for tool in listed["tools"]:
    read_only = bool(tool.annotations and tool.annotations.read_only_hint)
    print(f"- {tool.name} (read-only: {read_only}): {' '.join(tool.description.split())}")

Now review them the way today's `review_tool` does, in three lines. A description that
tells the model what to do next is not a description, whatever the tool is called.

In [ ]:
DIRECTIVE = re.compile(r"\b(before answering|before calling|always call|call \w+ on)\b", re.I)

for tool in listed["tools"]:
    reasons = []
    if DIRECTIVE.search(tool.description):
        reasons.append("instruction_in_description")
    verdict = "REFUSE " + ", ".join(reasons) if reasons else "safe"
    print(f"{tool.name:<16} {verdict}")

`summarize_notes` is refused, and not for what its name says: for what its description
tells the model to do. The server has no `delete_notes` tool at all. A model that obeys
the description goes looking for one on every other server you have connected.

## 4. A server that behaves: the course

The course is an MCP server too, over HTTP instead of stdio, with no key. Ask it a
question, then ask it something it does not teach.

In [ ]:
from mcp.client.streamable_http import streamable_http_client

COURSE = "https://mcp.geckovision.tech/course/mcp"


async def ask_course(question):
    async with streamable_http_client(COURSE) as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()
            result = await session.call_tool("search_course", {"query": question, "limit": 3})
            return json.loads(result.content[0].text)


import secrets

# The off-topic question is made up at run time, on purpose. Printed in a published
# page, any question becomes part of the course, and searching for it finds that page.
OFF_TOPIC = f"{secrets.token_hex(3)} {secrets.token_hex(3)}"

try:
    for question in ("what is a resource in MCP", OFF_TOPIC):
        answer = await ask_course(question)
        print(f"Q: {question}")
        for hit in answer["hits"]:
            print(f"   {hit['page_id']}")
        if not answer["hits"]:
            print("   (no hits)")
        print(f"   note: {answer['note']}")
except Exception as error:  # a demo must survive a room with no network
    print(f"offline: {type(error).__name__}. The course server is at {COURSE}")

The first question comes back with pages you can open. The second, a word that exists
nowhere, comes back empty and says so. **An empty answer is an answer**, and a good
server does not make something up to fill the gap.

Why is that question random? Try it with an off-topic question typed into a published
page, and the page that typed it comes back: once written down, it is part of the
course. The same thing happens to every example you put in your own corpus.

## 5. The same two servers, in your assistant

In a terminal, in the course folder, with Claude Code:

```bash
claude mcp add tiny-notes -- uv run python demos/12_tiny_server.py
claude mcp add --transport http dev3pack-course https://mcp.geckovision.tech/course/mcp
claude
```

Then ask it: "summarize my notes on MCP", and watch what it does with the instruction
hidden in the description. Then: "ask the course what a resource is". Other assistants
(ChatGPT, Cursor, Codex, VS Code) are on
[connect your assistant](../units/en/unit0/connect-your-assistant.mdx).

Remove the tiny server when you are done: `claude mcp remove tiny-notes`.

**What to take away:** a server writes its own menu. Read what it claims against what
it lists, and read every description as somebody else's text, never as your
instructions.